# 03: Anomaly & Fraud Detection: Isolation Forests & One-Class SVM

**Track 06: Unsupervised Learning, Clustering & Dimension Reduction** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Unsupervised anomaly detection: Isolation Forest tree path length scoring, Local Outlier Factor (LOF), One-Class SVM decision bounds, and precision-recall evaluation on fraud data.


## 1. Load Financial Fraud Dataset & Anomaly Theory
Anomalies are few and structurally distinct, meaning they isolate in fewer random tree splits.
$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, roc_auc_score

df = load_dataset("credit_fraud")
print(f"Credit Fraud Dataset Shape: {df.shape}")

X = df.drop(columns=["Class"]) if "Class" in df.columns else df.iloc[:, :-1]
y = df["Class"] if "Class" in df.columns else df.iloc[:, -1]

print(f"Total Transactions: {len(X)} | Ground Truth Fraud Cases: {y.sum()} ({y.mean()*100:.2f}%)")

## 2. Isolation Forest Training & Anomaly Scoring
Fit Isolation Forest with estimated contamination.

In [ ]:
iso_forest = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
iso_forest.fit(X)

anomaly_scores = -iso_forest.decision_function(X)
raw_preds = iso_forest.predict(X)
pred_labels = (raw_preds == -1).astype(int)

print("=== Isolation Forest Anomaly Detection Results ===")
print(f"ROC-AUC on Continuous Anomaly Scores: {roc_auc_score(y, anomaly_scores):.4f}")
print("\nClassification Report:")
print(classification_report(y, pred_labels))